# Treino do detector de EPI (capacete e óculos) — YOLO26n

Rode no Google Colab com GPU: **Ambiente de execução → Alterar o tipo de ambiente → GPU T4**.

Etapas: instalar dependências → baixar dataset do Roboflow → filtrar as 4 classes → treinar → avaliar → baixar `best.pt`.

Todas as células usam caminhos absolutos, então podem ser executadas de novo mesmo se a sessão for reiniciada.

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

## 1. Clonar o repositório (para usar o script de filtro)

In [ ]:
import os

REPO = "https://github.com/emersonscc/detector-EPI.git"  # ajuste aqui
ROOT = "/content/epi-detector"

if not os.path.isdir(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
print("Diretório atual:", os.getcwd())

## 2. Baixar o dataset PPEs do Roboflow
A API key fica em roboflow.com → Settings → API Keys. Ela é pedida na hora e não fica salva no notebook.

In [ ]:
import os
from getpass import getpass
from roboflow import Roboflow

ROOT = "/content/epi-detector"
os.chdir(ROOT)

if not os.path.isdir(f"{ROOT}/data/raw/ppes/train"):
    rf = Roboflow(api_key=getpass("Roboflow API key: "))
    project = rf.workspace("ramcomills").project("ppes-kaxsi-v4bep")
    project.version(1).download("yolov8", location=f"{ROOT}/data/raw/ppes")
else:
    print("Dataset já baixado.")

## 3. Filtrar para helmet / no_helmet / goggles / no_goggles

In [ ]:
ROOT = "/content/epi-detector"
!python {ROOT}/scripts/filter_dataset.py --src {ROOT}/data/raw/ppes --dst {ROOT}/data/processed/epi

## 4. Treinar
`yolo26n.pt` é baixado automaticamente com pesos pré-treinados no COCO (transfer learning).

In [ ]:
import os
from ultralytics import YOLO

ROOT = "/content/epi-detector"
os.chdir(ROOT)
DATA = f"{ROOT}/data/processed/epi/data.yaml"

model = YOLO("yolo26n.pt")
model.train(
    data=DATA,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,      # para cedo se não melhorar
    project=f"{ROOT}/runs",
    name="epi_yolo26n",
    exist_ok=True,
)
SAVE_DIR = str(model.trainer.save_dir)
print("Resultados salvos em:", SAVE_DIR)

## 5. Avaliar no conjunto de teste

In [ ]:
import glob, os
from ultralytics import YOLO

ROOT = "/content/epi-detector"
DATA = f"{ROOT}/data/processed/epi/data.yaml"
# acha o best.pt mesmo se a sessão foi reiniciada
SAVE_DIR = os.path.dirname(os.path.dirname(sorted(glob.glob(f"{ROOT}/runs/**/best.pt", recursive=True), key=os.path.getmtime)[-1]))
print("Usando:", SAVE_DIR)

best = YOLO(f"{SAVE_DIR}/weights/best.pt")
metrics = best.val(data=DATA, split="test")
print(f"\nmAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}")
for i, name in best.names.items():
    print(f"{name:>11}: mAP50-95 = {metrics.box.maps[i]:.3f}")

## 6. Ver algumas predições

In [ ]:
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob(f"{ROOT}/data/processed/epi/test/images/*"))[:6]
best.predict(imgs, conf=0.4, save=True, project=f"{ROOT}/runs", name="preview", exist_ok=True)
for p in sorted(glob.glob(f"{ROOT}/runs/**/preview/*.jpg", recursive=True))[:6]:
    display(Image(filename=p, width=500))

## 7. Baixar o modelo e os gráficos
Coloque o `best.pt` em `models/` no seu computador. Os gráficos (curvas, matriz de confusão) servem para o relatório.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/resultados_treino", "zip", SAVE_DIR)
files.download("/content/resultados_treino.zip")